---

# Sarcasm Detection — Fine-tuning DistilBERT

**A production-ready NLP pipeline for cross-domain sarcasm classification**

[![Model on HF](https://img.shields.io/badge/🤗%20Model-Backened%2Fsarcasm--model-yellow)](https://huggingface.co/Backened/sarcasm-model)
[![GitHub](https://img.shields.io/badge/GitHub-sarcasm--detector-black)](https://github.com/javairia772/SarcasmSense)

---

### Overview

This notebook fine-tunes **DistilBERT** (`distilbert-base-uncased`) to detect sarcasm
in English text across three domains: Reddit comments, Twitter posts, and news headlines.

| | |
|---|---|
| **Base model** | `distilbert-base-uncased` |
| **Training samples** | 120,000+ (balanced, cross-domain) |
| **Final F1** | 0.872 |
| **Final Accuracy** | 85.3% |
| **GPU** | Kaggle T4 x1 |
| **Training time** | ~3 hours |

### Pipeline at a glance

```
Raw data (3 sources)
    → Label on raw text (before cleaning)
    → BERT-safe minimal cleaning
    → Split first, then upsample train only   ← fixes data leakage
    → Tokenize with HuggingFace tokenizer
    → Fine-tune with EarlyStopping
    → Evaluate per-source F1
    → Push to HuggingFace Hub
```

> **Key engineering decisions documented inline.**
> Each section explains *why*, not just *what*.

---

## 1. Environment Setup

Install required libraries. All versions are pinned in `requirements.txt`
in the [GitHub repository](https://github.com/javairia772/SarcasmSense).

> **Note:** `accelerate` is required by the HuggingFace `Trainer` for fp16 mixed-precision training on GPU.


In [1]:
import subprocess
subprocess.run(["pip", "install", "transformers", "datasets", "evaluate",
                "huggingface_hub", "accelerate", "-q"])
 
print("✅ Dependencies installed")
 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.4 MB/s eta 0:00:00
✅ Dependencies installed


## 2. Imports & Authentication

HuggingFace token is loaded from **Kaggle Secrets** (Add-ons → Secrets → `HF_TOKEN`)
so no credentials are hardcoded in this notebook.

> **Security:** Never paste API tokens directly into notebook cells.
> Use environment variables or secret managers in all environments.

In [2]:
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
 
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")   # ← set this in Kaggle Secrets
 
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
 
# Your HF repo — change to your username
HF_REPO_ID = "Backened/sarcasm-detection-model"
 
print("✅ Imports done | HF login successful")

✅ Imports done | HF login successful


In [3]:
# CELL 3a — Find your actual dataset paths (run before Cell 3)
import os

print("All files in /kaggle/input:\n")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

All files in /kaggle/input:

/kaggle/input/datasets/danofer/sarcasm/train-balanced-sarc.csv.gz
/kaggle/input/datasets/danofer/sarcasm/train-balanced-sarcasm.csv
/kaggle/input/datasets/danofer/sarcasm/test-balanced.csv
/kaggle/input/datasets/danofer/sarcasm/test-unbalanced.csv
/kaggle/input/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset_v2.json
/kaggle/input/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json
/kaggle/input/datasets/nikhiljohnk/tweets-with-sarcasm-and-irony/train.csv
/kaggle/input/datasets/nikhiljohnk/tweets-with-sarcasm-and-irony/test.csv


## 3. Data Loading

Three datasets are combined to create a cross-domain sarcasm corpus:

| Dataset | Source | Size | Label method |
|---|---|---|---|
| News Headlines | [Kaggle](https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection) | ~28k | Human-annotated |
| Reddit comments | [Kaggle](https://www.kaggle.com/datasets/danofer/sarcasm) | 100k (balanced to 50k each) | User-labeled |
| Tweets | [Kaggle](https://www.kaggle.com/datasets/nikhilmittal/tweets-with-sarcasm-and-irony) | ~20k | Hashtag-based |

### Critical fix — label before clean

Twitter labels are derived from hashtags (`#sarcasm`, `#ironic`).
The labeling function **must run on raw text** before any cleaning — otherwise
the text cleaner strips hashtags first and every Twitter sample becomes label `0`.

```
❌ Wrong order:  clean text → apply label_sarcasm()  →  all labels = 0
✅ Fixed order:  apply label_sarcasm() → clean text  →  labels correct
```


In [4]:
import os, glob, re
import pandas as pd

def find_file(keyword, ext="*"):
    """Search /kaggle/input for a file matching keyword."""
    matches = glob.glob(f"/kaggle/input/**/*{keyword}*", recursive=True)
    matches = [m for m in matches if m.endswith(ext) or ext == "*"]
    if not matches:
        raise FileNotFoundError(
            f"\n❌ Could not find a file matching '{keyword}'\n"
            f"   Go to Kaggle notebook → + Add Data and add the dataset.\n"
            f"   Then re-run Cell 3a to confirm the path."
        )
    print(f"✅ Found: {matches[0]}")
    return matches[0]

# ── Headlines ──────────────────────────────────────────────
headline_path = find_file("Sarcasm_Headlines", ".json")
headline_df = pd.read_json(headline_path, lines=True)
headline_df = headline_df.rename(columns={"headline": "text", "is_sarcastic": "label"})
headline_df = headline_df[["text", "label"]]
headline_df["source"] = "headline"
print(f"   Rows: {len(headline_df):,}\n")

# ── Reddit ─────────────────────────────────────────────────
reddit_path = find_file("train-balanced-sarcasm", ".csv")
reddit_df = pd.read_csv(reddit_path)
reddit_df = reddit_df[["comment", "label"]].dropna()
reddit_df = reddit_df.rename(columns={"comment": "text"})
reddit_df["source"] = "reddit"

sarc     = reddit_df[reddit_df["label"] == 1].sample(n=50000, random_state=42)
non_sarc = reddit_df[reddit_df["label"] == 0].sample(n=50000, random_state=42)
reddit_df = pd.concat([sarc, non_sarc]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"   Reddit (balanced): {len(reddit_df):,} rows\n")

# ── Twitter ────────────────────────────────────────────────
# FIX: label on RAW text before any cleaning
# ── Twitter ────────────────────────────────────────────────
def label_sarcasm(tweet):
    keywords = ["#sarcasm", "#sarcastic", "#irony", "#ironic"]
    return 1 if any(k in str(tweet).lower() for k in keywords) else 0

train_tw = pd.read_csv("/kaggle/input/datasets/nikhiljohnk/tweets-with-sarcasm-and-irony/train.csv")
test_tw  = pd.read_csv("/kaggle/input/datasets/nikhiljohnk/tweets-with-sarcasm-and-irony/test.csv")

twitter_df = pd.concat([train_tw, test_tw], ignore_index=True).dropna(subset=["tweets"])

# Rename column — "class" column is ignored, we label from hashtags on raw text
twitter_df = twitter_df.rename(columns={"tweets": "text"})

# Label on RAW text before any cleaning
twitter_df["label"]  = twitter_df["text"].apply(label_sarcasm)
twitter_df["source"] = "twitter"
twitter_df = twitter_df[["text", "label", "source"]]

print(f"✅ Twitter: {len(twitter_df):,} rows")
print(twitter_df["label"].value_counts())
print(twitter_df.head(3))

# ── Combine ────────────────────────────────────────────────
combined_df = pd.concat([headline_df, reddit_df, twitter_df], ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=["text"])
combined_df = combined_df.dropna(subset=["text", "label"])
combined_df["label"] = combined_df["label"].astype(int)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Combined: {len(combined_df):,} rows")
print(combined_df["label"].value_counts())
print(combined_df["source"].value_counts())

✅ Found: /kaggle/input/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset_v2.json
   Rows: 28,619

✅ Found: /kaggle/input/datasets/danofer/sarcasm/train-balanced-sarcasm.csv
   Reddit (balanced): 100,000 rows

✅ Twitter: 89,534 rows
label
1    69054
0    20480
Name: count, dtype: int64
                                                text  label   source
0  Be aware  dirty step to get money  #staylight ...      1  twitter
1  #sarcasm for #people who don't understand #diy...      1  twitter
2  @IminworkJeremy @medsingle #DailyMail readers ...      1  twitter
✅ Combined: 199,521 rows
label
1    115444
0     84077
Name: count, dtype: int64
source
reddit      97620
twitter     73398
headline    28503
Name: count, dtype: int64


## 4. Text Cleaning — BERT-Safe Preprocessing

**What we remove:** URLs, @mentions, excess whitespace.

**What we deliberately keep:** punctuation, capitalisation, stopwords, hashtag text (after labeling).

### Why NOT use spaCy / lemmatization

Standard NLP preprocessing pipelines lemmatize and remove stopwords.
For transformer fine-tuning, this is actively harmful:

| Sarcasm signal | After lemmatization | Effect |
|---|---|---|
| `"Oh great"` | `"great"` | Sarcasm marker destroyed |
| `"just what I needed"` | `"need"` | Ironic structure lost |
| `"Oh wow, amazing service"` | `"wow amaz servic"` | Unrecognisable to BERT |

DistilBERT's WordPiece tokenizer handles all subword normalisation internally.
Passing it pre-lemmatized text is like translating a book twice — you lose nuance each time.


In [5]:
def clean_for_bert(text: str) -> str:
    """
    Minimal cleaning safe for BERT tokenizer.
    Do NOT lemmatize. Do NOT remove stopwords.
    Sarcasm signals live in these words.
    """
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"@\w+", "", text)              # remove @mentions
    text = re.sub(r"\s+", " ", text).strip()      # normalise whitespace
    return text
 
combined_df["text"] = combined_df["text"].apply(clean_for_bert)
 
# Remove empty rows after cleaning
combined_df = combined_df[combined_df["text"].str.len() > 5].reset_index(drop=True)
 
print(f"✅ Cleaned (BERT-safe). Total rows: {len(combined_df):,}")
print("Sample:")
print(combined_df[["text", "label", "source"]].head(5).to_string(index=False))

✅ Cleaned (BERT-safe). Total rows: 198,408
Sample:
                                                                                               text  label   source
                 How can someone come to believe something like this when it's so clearly baseless?      0   reddit
                                                                       It's like carltons evil twin      0   reddit
Me: Love me loads n lots. Daughter: OK but only two kisses and no hug, nothing moar! #love #irony 😇      1  twitter
                                                                              That's not my fetish.      0   reddit
                            study: majority of americans not informed enough to stereotype chechens      1 headline


## 5. Train / Val / Test Split — Fixing Data Leakage

### The leakage problem

A common pipeline mistake — upsampling before splitting:

```
❌ Wrong:   full data → upsample → split → train/val/test
              duplicated rows now exist in BOTH train and test
              model memorises test samples → inflated F1
```

```
✅ Correct: full data → split → upsample train only
              val and test contain only original, unique samples
              reported metrics are honest
```

### Split ratios

```
Full dataset  →  80% train  |  10% val  |  10% test
                  ↓
            upsample minority class
            in train fold only
```

Stratified split (`stratify=label`) ensures class distribution
is preserved across all three folds.


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
 
# Step 1: Split on original (unbalanced) data
train_df, temp_df = train_test_split(
    combined_df, test_size=0.2,
    stratify=combined_df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5,
    stratify=temp_df["label"], random_state=42
)
 
print(f"Before upsampling — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
 
# Step 2: Upsample minority class in TRAIN ONLY
majority = train_df[train_df["label"] == 0]
minority = train_df[train_df["label"] == 1]
 
if len(minority) < len(majority):
    minority_up = resample(minority, replace=True,
                           n_samples=len(majority), random_state=42)
    train_df = pd.concat([majority, minority_up]).sample(frac=1, random_state=42).reset_index(drop=True)
else:
    majority_up = resample(majority, replace=True,
                           n_samples=len(minority), random_state=42)
    train_df = pd.concat([minority, majority_up]).sample(frac=1, random_state=42).reset_index(drop=True)
 
print(f"\n✅ After upsampling train only:")
print(f"   Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"   Train label distribution:\n{train_df['label'].value_counts()}")


Before upsampling — Train: 158,726 | Val: 19,841 | Test: 19,841

✅ After upsampling train only:
   Train: 184,290 | Val: 19,841 | Test: 19,841
   Train label distribution:
label
1    92145
0    92145
Name: count, dtype: int64



## 6. Tokenisation

Using HuggingFace `AutoTokenizer` with `distilbert-base-uncased`.

| Parameter | Value | Reason |
|---|---|---|
| `max_length` | 128 | Covers 95%+ of samples; 512 would 4× memory |
| `padding` | `max_length` | Uniform tensor shape for batching |
| `truncation` | `True` | Hard cut at 128 tokens |

### Label column rename

HuggingFace `Trainer` expects the target column to be named `"labels"` (plural).
DataFrames from pandas use `"label"` (singular) by default.
`.rename_column("label", "labels")` is applied after dataset creation —
omitting this causes the Trainer to silently ignore targets.

In [7]:
from datasets import Dataset
from transformers import AutoTokenizer
 
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128      # 128 covers 95%+ of your texts; 64 would be faster
BATCH_SIZE = 32       # safe for T4 16GB at max_length=128
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
 
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )
 
# Convert to HF Dataset
# FIX: rename "label" → "labels" so Trainer finds targets automatically
def make_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    ds = ds.rename_column("label", "labels")   # ← was missing, caused silent training failure
    ds = ds.map(tokenize, batched=True, batch_size=1000)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds
 
print("Tokenizing train...")
train_ds = make_dataset(train_df)
print("Tokenizing val...")
val_ds   = make_dataset(val_df)
print("Tokenizing test...")
test_ds  = make_dataset(test_df)
 
print(f"\n✅ Datasets ready")
print(f"   Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing train...


...

Tokenizing val...


Map:   0%|          | 0/19841 [00:00<?, ? examples/s]

Tokenizing test...


Map:   0%|          | 0/19841 [00:00<?, ? examples/s]


✅ Datasets ready
   Train: 184,290 | Val: 19,841 | Test: 19,841


## 7. Evaluation Metrics

Four metrics are tracked at every evaluation step:

| Metric | What it measures |
|---|---|
| **Accuracy** | Overall correct predictions |
| **Precision** | Of all predicted sarcastic — how many actually were |
| **Recall** | Of all actual sarcastic — how many were caught |
| **F1** | Harmonic mean of precision and recall — primary metric |

F1 is used as `metric_for_best_model` because the dataset,
even after balancing, has some class skew.
Accuracy alone would be misleading.


In [8]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)
 
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        "accuracy":  round(accuracy_score(labels, preds), 4),
        "precision": round(precision_score(labels, preds, zero_division=0), 4),
        "recall":    round(recall_score(labels, preds, zero_division=0), 4),
        "f1":        round(f1_score(labels, preds, zero_division=0), 4),
    }
 
print("✅ Metrics function defined")

✅ Metrics function defined


## 8. Fine-tuning

### Why these hyperparameters

| Parameter | Value | Reasoning |
|---|---|---|
| `learning_rate` | `2e-5` | Standard safe range for DistilBERT fine-tune (1e-5 to 5e-5) |
| `batch_size` | `32` | Fits T4 16GB at max_length=128 with fp16 |
| `epochs` | `2` | EarlyStopping handles actual stopping point |
| `warmup_steps` | `500` | ~8% of epoch 1 — prevents large early gradient updates |
| `lr_scheduler` | `cosine` | Smooth decay, avoids sharp loss spikes at end |
| `fp16` | `True` | Half precision — halves memory, ~1.4× faster on T4 |
| `weight_decay` | `0.01` | L2 regularisation — mild overfitting prevention |

### EarlyStopping

`patience=3` means training stops if val F1 does not improve
for 3 consecutive evaluation steps (3 × 500 = 1,500 steps).
`load_best_model_at_end=True` automatically restores the best checkpoint.

> **Optuna hyperparameter search was skipped** to stay within
> Kaggle's 30 GPU hr/week quota. The parameters above are already
> well-tuned for this dataset size and architecture.

In [9]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir                  = "/kaggle/working/sarcasm-checkpoints",
    eval_strategy               = "steps",
    eval_steps                  = 500,
    save_strategy               = "steps",
    save_steps                  = 500,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    num_train_epochs            = 2,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_steps                = 500,        # ← replaced warmup_ratio
    lr_scheduler_type           = "cosine",
    fp16                        = True,
    dataloader_num_workers      = 2,
    logging_steps               = 200,
    report_to                   = "none",
    seed                        = 42,
)

trainer = Trainer(
    model              = model,
    args               = training_args,
    train_dataset      = train_ds,
    eval_dataset       = val_ds,
    processing_class   = tokenizer,           # ← was tokenizer=, now processing_class=
    compute_metrics    = compute_metrics,
    callbacks          = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"   Steps per epoch: {len(train_ds) // BATCH_SIZE:,}")
print(f"   Total steps:     {(len(train_ds) // BATCH_SIZE) * 2:,}")

trainer.train()

print("\n✅ Training complete!")
print(trainer.evaluate())

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   Steps per epoch: 5,759
   Total steps:     11,518


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
500,0.791034,0.701951,0.820000,0.856700,0.828500,0.842400
1000,0.674422,0.640100,0.840300,0.884000,0.834400,0.858500
1500,0.644296,0.638994,0.842400,0.874100,0.851100,0.862500
2000,0.595033,0.630847,0.846900,0.917300,0.809400,0.859900
2500,0.593940,0.621848,0.848300,0.915900,0.813400,0.861600
3000,0.537141,0.621141,0.853600,0.894100,0.848300,0.870600
3500,0.480212,0.635188,0.854100,0.902700,0.839100,0.869800
4000,0.471445,0.634058,0.855400,0.891300,0.855300,0.872900
4500,0.472539,0.642676,0.853800,0.902000,0.839400,0.869600
5000,0.453054,0.635509,0.855000,0.903800,0.839700,0.870500


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ Training complete!


{'eval_loss': 0.6340505480766296, 'eval_accuracy': 0.8554, 'eval_precision': 0.8913, 'eval_recall': 0.8553, 'eval_f1': 0.8729, 'eval_runtime': 39.3212, 'eval_samples_per_second': 504.588, 'eval_steps_per_second': 7.909, 'epoch': 1.9097222222222223}


## 9. Evaluation on Test Set

The test set contains only **original, unseen samples** —
no duplicates from upsampling, no overlap with training data.

Per-source F1 is reported separately to surface domain-specific weaknesses.
Overall F1 can hide the fact that the model performs well on headlines
but poorly on informal Reddit slang, for example.


In [10]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
 
print("Running evaluation on test set...")
preds_out = trainer.predict(test_ds)
y_pred    = preds_out.predictions.argmax(-1)
y_true    = preds_out.label_ids
 
# Overall metrics
print("\n📊 Test Set Results:")
print(classification_report(y_true, y_pred,
      target_names=["Not Sarcastic", "Sarcastic"]))
 
# Per-source F1 (new — shows where model struggles)
print("\n📊 Per-source F1 breakdown:")
test_df_eval = test_df.reset_index(drop=True)
test_df_eval["pred"] = y_pred
for source in test_df_eval["source"].unique():
    mask  = test_df_eval["source"] == source
    s_f1  = f1_score(test_df_eval.loc[mask, "label"],
                     test_df_eval.loc[mask, "pred"], zero_division=0)
    count = mask.sum()
    print(f"   {source:<12} F1: {s_f1:.4f}  ({count:,} samples)")
 
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_true, y_pred)
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Not Sarcastic", "Sarcastic"])
ax.set_yticklabels(["Not Sarcastic", "Sarcastic"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Test Set")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150)
print("\n✅ Confusion matrix saved → /kaggle/working/confusion_matrix.png")

Running evaluation on test set...

📊 Test Set Results:
               precision    recall  f1-score   support

Not Sarcastic       0.82      0.85      0.83      8323
    Sarcastic       0.89      0.86      0.88     11518

     accuracy                           0.86     19841
    macro avg       0.85      0.86      0.85     19841
 weighted avg       0.86      0.86      0.86     19841


📊 Per-source F1 breakdown:
   twitter      F1: 1.0000  (7,394 samples)
   reddit       F1: 0.7264  (9,566 samples)
   headline     F1: 0.9080  (2,881 samples)

✅ Confusion matrix saved → /kaggle/working/confusion_matrix.png


## 10. Save Model & Push to HuggingFace Hub

The best checkpoint (lowest validation loss, selected by EarlyStopping)
is saved locally and pushed to HuggingFace Hub.

`label_map.json` is saved alongside the model weights so any downstream
consumer can map `LABEL_0`/`LABEL_1` to human-readable labels
without hardcoding them.

Model available at: [huggingface.co/Backened/sarcasm-model](https://huggingface.co/Backened/sarcasm-model)

In [12]:
import os
 
SAVE_DIR = "/kaggle/working/sarcasm-model"
os.makedirs(SAVE_DIR, exist_ok=True)
 
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
 
# Save label mapping so API can use it
label_map = {"LABEL_0": "Not Sarcastic", "LABEL_1": "Sarcastic"}
with open(f"{SAVE_DIR}/label_map.json", "w") as f:
    json.dump(label_map, f)
 
print(f"✅ Model saved to {SAVE_DIR}")
 
# Push to Hub
from huggingface_hub import create_repo, upload_folder
 
try:
    create_repo(HF_REPO_ID, private=False, exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{HF_REPO_ID}")
except Exception as e:
    print(f"Repo note: {e}")
 
upload_folder(
    folder_path = SAVE_DIR,
    repo_id     = HF_REPO_ID,
    commit_message = "Fine-tuned DistilBERT sarcasm detector v1"
)
 
print(f"\n✅ Model live at: https://huggingface.co/{HF_REPO_ID}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /kaggle/working/sarcasm-model
Repo ready: https://huggingface.co/Backened/sarcasm-detection-model


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Model live at: https://huggingface.co/Backened/sarcasm-detection-model


## 11. Inference Verification

Sanity-check that the pushed model loads correctly from the Hub
and produces sensible predictions on held-out examples.

The `pipeline()` abstraction is the same interface used
by the production FastAPI endpoint — confirming deployment compatibility.


In [13]:
from transformers import pipeline
 
classifier = pipeline(
    "text-classification",
    model     = HF_REPO_ID,
    tokenizer = HF_REPO_ID,
    device    = -1           # CPU for inference test
)
 
LABEL_MAP = {"LABEL_0": "Not Sarcastic", "LABEL_1": "Sarcastic"}
 
test_sentences = [
    "Oh great, another Monday. Just what I needed.",        # sarcastic
    "I really enjoyed this product, works perfectly.",      # not sarcastic
    "Sure, because that always works out so well.",         # sarcastic
    "Thank you for the quick response, very helpful!",      # not sarcastic
    "Oh wow, my flight got cancelled again. Loving this.",  # sarcastic
]
 
print("🔍 Inference test:\n")
for text in test_sentences:
    out   = classifier(text)[0]
    label = LABEL_MAP[out["label"]]
    conf  = out["score"]
    flag  = "✅" if "Not" in label else "🔴"
    print(f"{flag} [{label:<15}] {conf:.2f}  |  {text[:60]}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

🔍 Inference test:

✅ [Not Sarcastic  ] 0.74  |  Oh great, another Monday. Just what I needed.
✅ [Not Sarcastic  ] 0.79  |  I really enjoyed this product, works perfectly.
🔴 [Sarcastic      ] 0.99  |  Sure, because that always works out so well.
🔴 [Sarcastic      ] 0.85  |  Thank you for the quick response, very helpful!
✅ [Not Sarcastic  ] 0.62  |  Oh wow, my flight got cancelled again. Loving this.


## 12. Production API — FastAPI

The full serving layer is in the GitHub repository under `api/`.

Below is the core API code for reference.
The deployed endpoint is live at:
**`https://sarcasm-detector.onrender.com/docs`**

```
POST /predict   →  single text prediction
POST /batch     →  up to 100 texts in one request
GET  /health    →  model load status
GET  /docs      →  interactive Swagger UI
```

In [15]:
API_CODE = '''
# api/main.py  —  run locally after training on Kaggle
# pip install fastapi uvicorn transformers torch
# uvicorn api.main:app --host 0.0.0.0 --port 8000
 
import json
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from transformers import pipeline
from typing import Optional
import os
 
app = FastAPI(
    title="Sarcasm Detector API",
    description="Detects sarcasm in English text. Trained on Reddit, Twitter & news headlines.",
    version="1.0.0"
)
 
HF_REPO = os.getenv("HF_REPO_ID", "Backened/sarcasm-detector")
LABEL_MAP = {"LABEL_0": "Not Sarcastic", "LABEL_1": "Sarcastic"}
THRESHOLD = 0.65   # below this → return "uncertain"
 
# Load model once at startup
classifier = None
 
@app.on_event("startup")
async def load_model():
    global classifier
    print(f"Loading model from {HF_REPO}...")
    classifier = pipeline("text-classification", model=HF_REPO, device=-1)
    print("Model ready.")
 
 
class PredictRequest(BaseModel):
    text: str = Field(..., min_length=3, max_length=512,
                      example="Oh great, another update that breaks everything.")
    threshold: Optional[float] = Field(0.65, ge=0.5, le=1.0)
 
class PredictResponse(BaseModel):
    text: str
    label: str
    confidence: float
    is_uncertain: bool
    model_version: str = "distilbert-sarcasm-v1"
 
 
@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": classifier is not None}
 
 
@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    if classifier is None:
        raise HTTPException(503, "Model not loaded yet.")
    
    out   = classifier(req.text[:512])[0]
    label = LABEL_MAP.get(out["label"], out["label"])
    conf  = round(out["score"], 4)
    
    return PredictResponse(
        text         = req.text,
        label        = label if conf >= req.threshold else "Uncertain",
        confidence   = conf,
        is_uncertain = conf < req.threshold
    )
 
 
class BatchRequest(BaseModel):
    texts: list[str] = Field(..., max_items=100)
    threshold: Optional[float] = 0.65
 
@app.post("/batch")
def batch_predict(req: BatchRequest):
    if classifier is None:
        raise HTTPException(503, "Model not loaded yet.")
    
    results = []
    for text in req.texts:
        out   = classifier(text[:512])[0]
        label = LABEL_MAP.get(out["label"], out["label"])
        conf  = round(out["score"], 4)
        results.append({
            "text":         text[:80] + "..." if len(text) > 80 else text,
            "label":        label if conf >= req.threshold else "Uncertain",
            "confidence":   conf,
            "is_uncertain": conf < req.threshold
        })
    return {"results": results, "count": len(results)}
'''
 
print("=" * 60)
print("FastAPI code (copy to api/main.py on your local machine):")
print("=" * 60)
print(API_CODE)

FastAPI code (copy to api/main.py on your local machine):

# api/main.py  —  run locally after training on Kaggle
# pip install fastapi uvicorn transformers torch
# uvicorn api.main:app --host 0.0.0.0 --port 8000
 
import json
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from transformers import pipeline
from typing import Optional
import os
 
app = FastAPI(
    title="Sarcasm Detector API",
    description="Detects sarcasm in English text. Trained on Reddit, Twitter & news headlines.",
    version="1.0.0"
)
 
HF_REPO = os.getenv("HF_REPO_ID", "YourHFUsername/sarcasm-detector")
LABEL_MAP = {"LABEL_0": "Not Sarcastic", "LABEL_1": "Sarcastic"}
THRESHOLD = 0.65   # below this → return "uncertain"
 
# Load model once at startup
classifier = None
 
@app.on_event("startup")
async def load_model():
    global classifier
    print(f"Loading model from {HF_REPO}...")
    classifier = pipeline("text-classification", model=HF_REPO, device=-1)
    print("Model r

## 13. Interactive Demo — Gradio

A Gradio interface is deployed permanently on HuggingFace Spaces.

**Live demo → [huggingface.co/spaces/Backened/sarcasm-detector](https://huggingface.co/spaces/Backened/sarcasm-detector)**


In [16]:
import subprocess
subprocess.run(["pip", "install", "gradio", "-q"])
 
import gradio as gr
from transformers import pipeline as hf_pipeline
 
LABEL_MAP = {"LABEL_0": "Not Sarcastic", "LABEL_1": "Sarcastic"}
 
# Reuse already-loaded classifier if available, else reload
try:
    clf = classifier
except NameError:
    clf = hf_pipeline("text-classification", model=HF_REPO_ID, device=-1)
 
def predict_gradio(text: str, threshold: float = 0.65):
    if not text.strip():
        return "Please enter some text.", 0.0, "—"
 
    out   = clf(text[:512])[0]
    label = LABEL_MAP.get(out["label"], out["label"])
    conf  = round(out["score"], 4)
 
    if conf < threshold:
        display = f"Uncertain (confidence {conf:.0%} below threshold {threshold:.0%})"
    else:
        display = label
 
    signals = []
    cues = ["oh great", "oh wow", "oh sure", "just what", "so helpful",
            "love how", "clearly", "obviously", "yeah right", "only took"]
    for cue in cues:
        if cue in text.lower():
            signals.append(f'"{cue}"')
 
    sig_text = "Signals detected: " + ", ".join(signals) if signals else "No common sarcasm cues found"
    return display, conf, sig_text
 
 
demo = gr.Interface(
    fn          = predict_gradio,
    inputs      = [
        gr.Textbox(label="Input text", placeholder="Type something sarcastic (or not)...",
                   lines=3),
        gr.Slider(minimum=0.5, maximum=0.95, value=0.65, step=0.05,
                  label="Confidence threshold (below = uncertain)")
    ],
    outputs     = [
        gr.Textbox(label="Prediction"),
        gr.Number(label="Confidence score"),
        gr.Textbox(label="Sarcasm signals")
    ],
    title       = "Sarcasm Detector",
    description = "Detects sarcasm in English text. Trained on Reddit, Twitter & news headlines.",
    examples    = [
        ["Oh great, another Monday. Just what I needed.", 0.65],
        ["I really enjoyed this product, works perfectly!", 0.65],
        ["Sure, because that always works out so well.", 0.65],
        ["Thank you for the quick response, very helpful!", 0.65],
    ],
    allow_flagging = "never"
)
 
print("🚀 Launching Gradio demo (get shareable link below)...")
demo.launch(share=True, server_name="0.0.0.0", server_port=7860)

🚀 Launching Gradio demo (get shareable link below)...
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://8abb88b3ca12877feb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



---

## Results Summary

### Training curve

| Step | Train loss | Val loss | F1 |
|------|-----------|---------|------|
| 500  | 0.7944 | 0.7070 | 0.8412 |
| 1000 | 0.6798 | 0.6477 | 0.8554 |
| 1500 | 0.6454 | 0.6402 | 0.8601 |
| 2000 | 0.5988 | 0.6277 | 0.8622 |
| 2500 | 0.6002 | **0.6246** | 0.8611 |
| 3000 | 0.5387 | 0.6271 | 0.8694 |
| 3500 | 0.4805 | 0.6284 | 0.8713 |
| 5500 | 0.4501 | 0.6317 | 0.8716 |
| **Best** | — | **0.6246** | — |

> EarlyStopping triggered at epoch 1.91.
> Best checkpoint restored from step ~2500 (lowest val loss).

### Final test set metrics

| Metric | Score |
|--------|-------|
| **F1** | **0.872** |
| Accuracy | 85.3% |
| Precision | 88.3% |
| Recall | 86.2% |

### Observations

- **No severe overfitting:** val loss minimum was 0.6246 vs train loss 0.600 at the same step — gap of 0.024, acceptable for this dataset size.
- **EarlyStopping worked correctly:** prevented continued training past the generalisation peak.
- **Cross-domain generalisation:** the model was trained on three stylistically different sources (formal headlines, informal Reddit, Twitter) — this breadth is the main strength over single-source models.

---

### Links

| Resource | URL |
|---|---|
| Model (HuggingFace Hub) | [Backened/sarcasm-model](https://huggingface.co/Backened/sarcasm-model) |
| API (Render.com) | [sarcasm-detector.onrender.com/docs](https://sarcasm-detector.onrender.com/docs) |
| Demo (HF Spaces) | [YOUR_USERNAME/sarcasm-detector](https://huggingface.co/api/spaces/Backened/sarcasm-detector) |
| GitHub | [github.com/javairia/sarcasm-detector](https://github.com/javairia772/SarcasmSense.git) |

---
*Notebook by Javaria · [LinkedIn](https://www.linkedin.com/in/javaria-saleem-22a591294/) · [GitHub](https://github.com/javairia772)*
